### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so the user is asking why parrots talk. Let me start by recalling what I know about parrots and their behavior. Parrots are known for their ability to mimic sounds, including human speech. But why do they do that? I think it might have something to do with their social behavior. In the wild, parrots live in groups, and vocalizations play a big role in communication. So, maybe they mimic sounds to integrate into the group or to establish their identity.\n\nI remember reading that parrots have a part of their brain called the song system, which is involved in vocal learning. This is similar to how songbirds learn songs. But how does that relate to mimicking human speech? Maybe it\'s a form of social bonding. If they\'re kept as pets, they might mimic their owners\' speech to interact or gain attention. Also, some parrots have been observed repeating phrases they hear around them, which could be a way to communicate with their human companions.\n\nAnother

In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [3]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. Let me check what tools I have available. There\'s a function called get_weather that takes a location parameter. Since the user specified Boston, I need to call that function with "Boston" as the location. I\'ll make sure to format the tool call correctly within the XML tags. No other functions are available, so this should be straightforward. Let me structure the JSON with the function name and the arguments as required.\n', 'tool_calls': [{'id': 'yy0ts2tyt', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 120, 'prompt_tokens': 154, 'total_tokens': 274, 'completion_time': 0.196837385, 'completion_tokens_details': {'reasoning_tokens': 96}, 'prompt_time': 0.006287307, 'prompt_tokens_details': None, 'queue_time': 1.1518792900000001, 'total_time': 0.203124692}, 'model_name': 'qwen

### tool execution loop

In [4]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather in Boston is sunny today! ☀️ You can expect clear skies and no rain, making it a great day for outdoor activities.


In [5]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified Boston, I need to call this function with "Boston" as the location. I\'ll make sure to format the tool call correctly within the XML tags as instructed.\n', 'tool_calls': [{'id': 's2q58kbxf', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 153, 'total_tokens': 246, 'completion_time': 0.136684727, 'completion_tokens_details': {'reasoning_tokens': 69}, 'prompt_time': 0.008515751, 'prompt_tokens_details': None, 'queue_time': 0.050636709, 'total_time': 0.145200478}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_deman